In [1]:
import math
import pandas as pd

df = pd.read_csv("./books_dataset.csv")
#df["isbn13"].dtype
#df.head()

## 1. using classes and dictionary --> Weighted Sum Model
* can't do a multiple linear regression since I don't have the users' history to compare to

### Loading and Cleaning
* used regex pattern method which I found in a stack overflow forum
* I found it while trying to clean up title columns but I learned that this could also be applied to cleaning up genres
* the forum: https://stackoverflow.com/questions/20894525/how-to-remove-parentheses-and-all-data-within-using-pandas-python
* a wesbite going into full detail about regex: https://brbgg.com/regex/
  * str.replace(r"\(.*?\)", "", regex = True)
    * r"" indicates that we are looking at a raw string
    * \( matches the opening parentheses
    * . matches any character, * tells it to match the number of character zero or more times
    * ? tells it to not be greedy, meaning it would continue deleting everything until it hit another )
      * if i had two sets of parentheses, without ?, it would delete everything. For example, if i had a string that was like x (12) and y (34), then without ?, the computer would delete (12) and y (34)


In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("books_dataset.csv")

#cleaning up the names of the columns so there is no unexpected whitespace
df.columns = df.columns.str.strip()

#get rid of potential Na/NaN values so that we can compute the math and copied the data set 
copy_df = df.dropna(subset = ['isbn13', 'title', 'genres', 'avg_rating', 'ratings_count']).copy()

copy_df["isbn13"] = copy_df["isbn13"].astype(str)

#altering title column so it drops paranthesis and everything inside
#copy_df["title"] = copy_df["title"].astype(str).str.replace(r"\(.*?\)", "", regex = True).str.strip()

copy_df["title"] = copy_df["title"].astype(str).str.strip()


## altering the genres to only display the one genre (just going to grab the first for simplicity)
### .astype(str) ensure that every value in column is treated as plain text to avoid crashes
### str.replace(r"[\[\]\"']", "", regex = True) using regex pattern 
#r".." = raw string, \ tells it look for those symbols in text
copy_df["genres"] = (
    copy_df["genres"]
        .astype(str)
        .str.replace(r"[\[\]\"']","", regex = True)
        .str.split(",")
        .str[0]
        .str.strip()
        .str.lower()
)

#print(copy_df[["title","genres"]].head(10))
#copy_df[["title", "genres"]].head(10)
copy_df.head()
#print(len(copy_df)) #68,119 rows 
#copy_df.columns

,isbn13,title,author,description,genres,avg_rating,ratings_count,publisher,format,pages,pub_year,series_name,series_position
0,9780439023481,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,Winning will make you famous.\nLosing means ce...,young-adult,4.34,4899965.0,Scholastic Press,Hardcover,374.0,2008.0,The Hunger Games,1.0
1,9780439554930,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling,Harry Potter's life is miserable. His parents ...,fantasy,4.45,4765497.0,Scholastic Inc,Hardcover,320.0,1997.0,Harry Potter,1.0
2,9780316015844,"Twilight (Twilight, #1)",Stephenie Meyer,About three things I was absolutely positive.\...,fantasy,3.57,3941381.0,"Little, Brown and Company",Paperback,498.0,2006.0,Twilight,1.0
3,9781649374042,"Fourth Wing (The Empyrean, #1)",Rebecca Yarros,A #1 New York Times bestseller • TV series in ...,fantasy,4.56,3818964.0,Entangled: Red Tower Books,NaN,517.0,2023.0,NaN,NaN
4,9780061120084,To Kill a Mockingbird,Harper Lee,The unforgettable novel of a childhood in a sl...,fiction,4.26,3255518.0,Harper Perennial Modern Classics,Paperback,324.0,2006.0,To Kill a Mockingbird,NaN


### Book class
* has the attributes that the program will need and store in the inventory dictionary that will be used throughout to look up variables 

In [3]:
class Book:
    def __init__(self, title, genres, raw_rating, ratings_count):
        self.title = title
        self.genres = genres
        self.raw_rating = float(raw_rating)
        self.ratings_count = int(ratings_count)

        # initializing the rating related variables for later calculations
        # going to normalize the variables so created place holders for later
        self.norm_rating = 0.0
        self.norm_reviews = 0.0


    # #had AI check what was going on and turns out I need to add the function
    # #it categorized this as a lookup function
    # # it looks up a book's isolated genre tag within the users profile, it finds the match and gets the gof
    
    # def proportions(self, user_profile):
    #     return user_profile.prop_genres.get(self.genres, 0.0)


# creating the dictionary of books with those variables and feeding it into the book class
inventory = {}

# index helps the computer know what row number its on
# row is helping computer look at entire row
for index, row in copy_df.iterrows():
    #isbn instead of title for key
    inventory[row["isbn13"]] = Book(
        title = row["title"],
        genres = row["genres"],
        raw_rating = row["avg_rating"],
        ratings_count = row["ratings_count"]
    )

#59991 when key was title --> key is isbn13, have all 68,119
print(f"total books in inventory: {len(inventory)}") 
#print(f"checking inventory: {inventory["The Hobbit"].genres}")
print(dict(list(inventory.items())[:4]))

total books in inventory: 68119
{'9780439023481': <__main__.Book object at 0x108579fd0>, '9780439554930': <__main__.Book object at 0x10ec5bc50>, '9780316015844': <__main__.Book object at 0x10ec5b9d0>, '9781649374042': <__main__.Book object at 0x10ed02b10>}


### Normalization function
* applying the min-max normalization
* for ratings_count, need to apply log-transformation to make the values smaller before applying the min-max normalization
* this function is what fills the current norm_rating and norm_reviews placeholders that are in the Book Class

In [4]:
def normalization(inventory):
    
    # finding the min/max of raw_ratings directly in inventory.values()
        # books.raw_rating is telling computer to only pull out the raw_rating and look through that Book object
    min_rating = min(book.raw_rating for book in inventory.values())
    max_rating = max(book.raw_rating for book in inventory.values())

    # before finding the min and max, need to apply log transformation on count_rating
    # converting list of the ratings counts of books into a numpy array and runs the log transformation ln(1+x)
    log_counting = np.log1p([book.ratings_count for book in inventory.values()])

    min_reviews = log_counting.min()
    max_reviews = log_counting.max()

    #looping over the inventory dictionary values and applying the normalization
    #had AI check and it said to add the else statement just in case the books has the exact same ratings or reviews 
    # if book has exact same rating and reviews then the denominator will be zero, trying to avoid dividing by zero
    for book in inventory.values():
        # if max is greater than min rating, compute the normalization else, have it default to 1 to avoid division by 0
        if max_rating > min_rating:
            book.norm_rating = (book.raw_rating - min_rating) / (max_rating - min_rating)
        else:
            book.norm_rating = 1.0

        if max_reviews > min_reviews:
            book.norm_reviews = (np.log1p(book.ratings_count) - min_reviews) / (max_reviews - min_reviews)
        else:
            book.norm_reviews = 1.0
    
normalization(inventory)
#print(vars(inventory["The Hobbit"]))

### User class
* takes the user inputs
* computes a proportion parameter that will then apply to the rest of the dictionary (inventory)
  * this proportion is our relative frequency

In [5]:
class User:
    def __init__(self, user_isbns, inventory):
        self.user_isbns = user_isbns

        #empty list to store the genres of the books that the user inputed
        self.fav_genres = []

        #grabbing the genre of the titles that user inputed and putting them into list
        for isbn in self.user_isbns:
            if isbn in inventory:
                #grabbing all of the book related objects
                book = inventory[isbn] 
                #only grabbing genres from book and appending into list
                self.fav_genres.append(book.genres) 

        self.prop_genres = {}
        
        #in order for computer to know to use the function, need to "call it"
        self.proportions()
        
    #calculating the proportions to get the relative frequency
    def proportions(self):
        
        #grabbing the total number of genres which will be the denominator
        total_genres = len(self.fav_genres)

        #counting the amount of genres which are the same
        for genre in self.fav_genres:
            count = self.fav_genres.count(genre)

            #getting the final proportion and storing into the self.prop_genres dictionary
            self.prop_genres[genre] = count/total_genres

#hard coded isbn numbers as user input:
#9780439023481	The Hunger Games (The Hunger Games, #1)
#9780439554930	Harry Potter and the Sorcerer's Stone (Harry Potter, #1)
#9780316015844	Twilight (Twilight, #1)

user_input = ["9780439023481", "9780439554930", "9780316015844"]
active_user = User(user_input, inventory)

print(f"Extracted genres: {active_user.fav_genres}")
print(f"Calculated proportions: {active_user.prop_genres}")

Extracted genres: ['young-adult', 'fantasy', 'fantasy']
Calculated proportions: {'young-adult': 0.3333333333333333, 'fantasy': 0.6666666666666666}


### Scoring function 
* this is where the weighted sum model gets constructed
  * going to need the proportions that were computed in the user class and need to set the beta values
* to get the top five recommendations, I used the dictionary method
  * resources that I used:
    * to help sort: https://www.youtube.com/watch?v=cd-vtiO5chk&t=208s
    * i did use AI to check to make sure I was sorting correctly and slicing correctly and how to get the output to be visibly nice 

In [6]:
def scoring_function(user, inventory, beta_prop = 0.5, beta_rating = 0.3, beta_reviews = 0.2):

    scored_recs = {}

    #inventory.items gives us (isbn, book object) so need to look in both hence isbn, book
    for isbn, book in inventory.items():
        # we don't want to recommend a book that user has already read (aka inputted)
        #user is us looking into class user and .user_isbns is us looking specifically in that list
        if isbn in user.user_isbns:
            continue

        # prop = book.proportions(user) #to use, need to add function in book class
        
        #looking up the proportion directly from user.prop_genres (user class)
        prop = user.prop_genres.get(book.genres, 0.0)

        #computing that titles score and storing it into the dictionary
        scored_recs[isbn] = (beta_prop * prop) + (beta_rating * book.norm_rating) + (beta_reviews * book.norm_reviews)
    
    #items methods returns each pair into a tuple (isbn, score)
    #for every pair of items in dicitonary, take each pair, return the second element (score), return element at index 1 (the item)
    #take tuple, lambda item:item[1] tells us to focus on second element (score)
    #sorted sorts by second element, dict converts tuples into dictionary
    top_five_recs = dict(sorted(scored_recs.items(), key= lambda item: item[1], reverse = True)[:5])

    #print(f"top five recs: {top_five_recs}")
    
    #used AI to also get the recommendations to print because I wasn't sure how to have it sift through everything to get the top five
    for rank, (isbn, score) in enumerate(top_five_recs.items(), start = 1):
        book = inventory[isbn]
        print(f"{rank} : {book.title}")
        #print(f"Model Regression Score Y = {score:.4f}")
        print(f"Genre: {book.genres} | Rating: {book.raw_rating:.2f} | Score: {score:.4f}\n")
    
scoring_function(active_user, inventory)

1 : Fourth Wing (The Empyrean, #1)
Genre: fantasy | Rating: 4.56 | Score: 0.7809

2 : Harry Potter and the Deathly Hallows (Harry Potter, #7)
Genre: fantasy | Rating: 4.62 | Score: 0.7772

3 : Harry Potter and the Half-Blood Prince (Harry Potter, #6)
Genre: fantasy | Rating: 4.54 | Score: 0.7678

4 : Harry Potter and the Goblet of Fire (Harry Potter, #4)
Genre: fantasy | Rating: 4.53 | Score: 0.7672

5 : Harry Potter Boxset (Harry Potter, #1-7)
Genre: fantasy | Rating: 4.74 | Score: 0.7604



### Interactive chatbot
commented this section out since its still in the works!!

In [7]:
# print("Welcome! Looking for your next read? Let me help!")
# print("Enter 3 books you enjoyed!")

# user_input = []

# while len(user_input) < 3:
#     given_title = input(f"\nBook #{len(user_input) + 1}: ").strip().lower()
    
#     for isbn, book in inventory.items():
#         clean_title = book.title.split("(")[0].strip().lower()
#         if given_title == clean_title:
            
#             found_isbn = isbn
#             matched_title = book.title
#             break #has computer stop searching once a match was found
    

# active_user = User(user_input, inventory)

# def main():
#     print("Welcome! Looking for your next read? Let me help!")

#     while True:
#         print("\nMenu")
#         print()
#         user_input = input()


dictionary of classes, 
prompt user to type in user name or find candidates
0 or 1 or multiple

In [8]:
# def isnum(n):
#     for i in n:
#         if i.isdigit() == False and i != "." and i != "-":
#             return False
#     return True

In [9]:
# count = 5

# while count >= 1:

#     num = input( "Give me a number (type 1) or a letter (type 2): " )
#     count -= 1
#     if num == "1":
#         print("Okay, user choose a number. Next step...")
#         if isnum(num):
#             num = float(num)
#             print("YES! We got a number:", num)
#             break
#         else:
#             print("Not a number!")

#     else:
#         print("Okay, user choose a letter. Next step...")


# if count == 0:
#     print("Too many tries...")